# 01 · Render the labelled image dataset (Colab, GPU)

Runtime → Change runtime type → **T4 GPU**. Renders synthetic Nightingale terrains with Blender Cycles, with pixel-exact hazard labels.

- train/val terrains: cleaned real ground + injected power-law boulders (synthetic)
- test terrain: the **real** Nightingale surface (held out, never trained on)

The run is resumable: finished frames are skipped, and everything is copied to Google Drive.

In [ ]:
# 1. Blender 5.2.2 (same version as local) and its headless runtime libraries
!apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null
!wget -q https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz -O /content/blender.tar.xz
!mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!/content/blender/blender --version | head -1

In [ ]:
# 2. Code and Python deps
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /content/touch_down || (cd /content/touch_down && git pull -q)
%cd /content/touch_down
!pip install -q -e .
import os
os.environ['TOUCHDOWN_DATA'] = '/content/data'
os.environ['TOUCHDOWN_BLENDER'] = '/content/blender/blender'

In [ ]:
# 3. NASA Nightingale 5 cm model -> height grid (about 3 minutes, once)
!python scripts/fetch_data.py
!python scripts/build_dtm.py

In [ ]:
# 4. Mount Drive so progress survives a disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/touchdown/dataset'
!mkdir -p {OUT}

In [ ]:
# 5. Render. About 4,500 frames in total; re-run this cell after a disconnect and it resumes.
!python -m touchdown.dataset.generate --out {OUT} --split test  --terrains 0 1  --frames 300 --device GPU
!python -m touchdown.dataset.generate --out {OUT} --split val   --terrains 0 5  --frames 40  --device GPU
!python -m touchdown.dataset.generate --out {OUT} --split train --terrains 0 40 --frames 100 --device GPU

In [ ]:
# 6. Sanity check
import json, collections
rows = [json.loads(l) for l in open(f'{OUT}/meta.jsonl')]
print(collections.Counter(r['split'] for r in rows))
print('mean boulder fraction:', sum(r['frac_boulder'] for r in rows) / len(rows))